### Import Dependencies

Run `uv sync` from the project root. Start Qdrant and run `02-RAG-preprocessing-items.ipynb` first to populate `Amazon-items-minilm-l6-v2`. Embeddings run locally; Gemini answer generation requires a Google API key and available project quota.


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

# Locate the project whether the kernel starts in the repo or notebook folder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel inside the project directory.")
load_dotenv(PROJECT_ROOT / ".env", override=True)

COLLECTION_NAME = "Amazon-items-minilm-l6-v2"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")


/Users/ganesh.m/Projects/ai-engineering-bootcamp/project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Local embedding function

Use the same model and normalization as notebook 02. The first run downloads the model; later runs use its cache. MiniLM produces 384-dimensional vectors and truncates inputs beyond 256 word pieces.


In [2]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def get_embedding(text):
    return embedding_model.encode(text, normalize_embeddings=True).tolist()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7715.35it/s]


### Retrieval function

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")
if not qdrant_client.collection_exists(COLLECTION_NAME):
    raise RuntimeError(
        f"Collection {COLLECTION_NAME!r} is missing. "
        "Run 02-RAG-preprocessing-items.ipynb first."
    )


In [4]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [5]:
retrieved_context = retrieve_data("what kind of earphones can you offer?", qdrant_client, k=10)

In [6]:
retrieved_context

{'retrieved_context_ids': ['B09KQP2H7N',
  'B0CH8DRD6K',
  'B0BBF2VC6X',
  'B09X9838WY',
  'B0BRV544MV',
  'B09TFM1SFQ',
  'B0BM657X74',
  'B0BC4PGXFK',
  'B0CFHWF326',
  'B0C996WY16'],
 'retrieved_context': ["Kids Wireless Headphones, Adjustable Headband, Stereo Sound, 3.5mm Jack, Kids Bluetooth Headphones, Volume Control, Foldable, Build-in Microphone, Over-Ear Headphones for Kids for School Home, Yellow 【WIRELESS & WIRED KIDS HEADPHONES】 Stunning kid headphones, built with the latest high quality 5.0 Bluetooth chip, which allows kids wireless headphones to connect fast and stable, also with 3.5mm jack to connect any device. You can use them as wireless headphones and wired headphones, compatible with smartphones, laptops, tablets, computers, TVs and many other Bluetooth-enabled devices, making it possible to provide you with great music anywhere in the occasion. 【Cute cat headphones】It is designed with cute cartoon pattern, you can use it in any venue. Our over-ear headphones provid

### Format retrieved context function

In [7]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [8]:
preprocessed_context = process_context(retrieved_context)

In [ ]:
print(preprocessed_context)

### Create Prompt template function

In [9]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [10]:
prompt = build_prompt(preprocessed_context, "what kind of earphones can you offer?")

In [11]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- ID: B09KQP2H7N, rating: 4.2, description: Kids Wireless Headphones, Adjustable Headband, Stereo Sound, 3.5mm Jack, Kids Bluetooth Headphones, Volume Control, Foldable, Build-in Microphone, Over-Ear Headphones for Kids for School Home, Yellow 【WIRELESS & WIRED KIDS HEADPHONES】 Stunning kid headphones, built with the latest high quality 5.0 Bluetooth chip, which allows kids wireless headphones to connect fast and stable, also with 3.5mm jack to connect any device. You can use them as wireless headphones and wired headphones, compatible with smartphones, laptops, tablets, computers, TVs and many other Bluetooth-enabled devices, making it possible to provide you with gre

### Generate an answer with Gemini

Uses stable `gemini-3.5-flash-lite`, optimized for high-throughput workloads. Set `GOOGLE_API_KEY` (or `GEMINI_API_KEY`) in the project `.env`. Optionally set `GEMINI_MODEL` to another model your project can access. Access and rate limits depend on your project; availability is not guaranteed.

Model documentation: https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite


In [12]:
google_api_key = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()
if not google_api_key:
    raise RuntimeError("Set GOOGLE_API_KEY or GEMINI_API_KEY in the project .env.")
gemini_client = genai.Client(api_key=google_api_key)

def generate_answer(prompt):
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    if not response.text:
        raise RuntimeError("Gemini returned no text. Check the response for blocked content.")
    return response.text


In [13]:
print(generate_answer(prompt))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


We offer a variety of earphones among the available products, including:

Kids wireless and wired headphones with volume control and built-in microphones (ID: B09KQP2H7N and ID: B0BBF2VC6X)
Apple MFi certified iPhone wired earbuds with 3.5mm jacks and microphones (ID: B0CH8DRD6K)
Wireless Bluetooth earbuds with charging cases, waterproof designs, and touch controls (IDs: B09X9838WY, B0BRV544MV, and B09TFM1SFQ)
Bluetooth sleep headband headphones that double as eye masks and sports headbands (ID: B0BM657X74 and ID: B0CFHWF326)
Running headphones designed with a neckband and behind-the-neck ear hook design (ID: B0BC4PGXFK)


### Combined RAG Pipeline

In [14]:
def rag_pipeline(question, top_k=5):
    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    return generate_answer(prompt)


In [15]:
print(rag_pipeline("what kind of earphones can you offer?", 10))

We offer a variety of available products including kids wireless and wired headphones, iPhone wired headphones with a 3.5mm jack, Volkano kids wired earbuds with a safe volume limit, Jesebang wireless earbuds with touch control, Wekily Bluetooth 5.3 wireless earbuds, Pamu active noise cancelling wireless earbuds, sleep headphones headbands, RUNAR running neckband earphones, MUSICOZY Bluetooth sports headbands, and Raymate portable waterproof Bluetooth speakers.


In [16]:
print(rag_pipeline("what kind of earphones can you offer? I am only interested in the ones with a rating of 4 and above.", 10))

Based on the available products, here are the earphones with a rating of 4 and above:

Volkano Ninja Kids Earphones for Boys with Carry Case and Keyring (ID: B0BBF2VC6X, rating: 4.2)
2 Pack iPhone Wired Headphones with 3.5mm Jack (ID: B0CH8DRD6K, rating: 4.9)
Kids Wireless Headphones, Adjustable Headband, Stereo Sound, 3.5mm Jack (ID: B09KQP2H7N, rating: 4.2)
Wekily Bluetooth 5.3 Headphones, Wireless Earbuds with 40H Playtimes Charge Case (ID: B0BRV544MV, rating: 4.4)
Jesebang Wireless Earbud, Bluetooth 5.2 Headphones Touch Control (ID: B09X9838WY, rating: 4.2)
Sleep Headphones Bluetooth Headband Wireless (ID: B0BM657X74, rating: 4.6)
MUSICOZY Bluetooth 5.3 Headband Headphones Built-in ENC Mic (ID: B0CFHWF326, rating: 4.1)
LONGFUN Cleaning Pen Kit for Airpods Pro (ID: B09R1S39N5, rating: 4.4)
